# HONEST-RL-Calibrator — GRPO Training Notebook

**Goal:** Teach `Qwen2.5-3B-Instruct` to be *calibrated* — confident when right, uncertain when wrong.

> ⚙️ Recommended runtime: **A100 GPU** (17GB+ VRAM). T4 works but is ~3× slower.

---

## Step 1 — Install Dependencies

In [ ]:
# Install Unsloth first (must come before trl/transformers)
!pip install unsloth --quiet
!pip install -U trl peft transformers datasets bitsandbytes accelerate aiohttp python-constraint --quiet
print('✓ Dependencies installed')

## Step 2 — Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/HONEST-Env.git'  # ← update this

if not os.path.exists('/content/HONEST-Env'):
    !git clone {REPO_URL} /content/HONEST-Env
else:
    !git -C /content/HONEST-Env pull

%cd /content/HONEST-Env
print('✓ Repository ready')

## Step 3 — Configure Environment Variables

In [ ]:
import os

# ── Required: HuggingFace token for Qwen2.5-3B-Instruct ─────────────────────
os.environ['HF_TOKEN'] = ''         # get from https://huggingface.co/settings/tokens

# ── Optional: deployed HONEST-Env HF Space (leave empty for local reward) ───
os.environ['HONEST_ENV_URL'] = ''   # e.g. 'https://your-space.hf.space'

# ── Optional: W&B key for experiment tracking ────────────────────────────────
os.environ['WANDB_API_KEY'] = ''    # get from https://wandb.ai/authorize

print('✓ Environment variables set')

## Step 4 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 5 — Dry Run (Sanity Check)

Runs all four reward functions against dummy completions without loading the full model.  
Expected output:
```
[correct]  brier=-0.0100  format=0.0500  accuracy=0.1000  anti_hedge=0.0000
[hedge]    brier=-0.2300  format=0.0500  accuracy=0.1000  anti_hedge=-0.0700
[malformed] brier=-0.5000 format=0.0000  accuracy=0.0000  anti_hedge=0.0000
```

In [ ]:
!python training/train_grpo.py --dry-run

## Step 6 — Baseline Evaluation (Before GRPO)

Establishes the pre-training baseline on Qwen2.5-3B-Instruct.  
Results saved to `eval/baseline_results.json` — **do not overwrite this file later**.  
Estimated time: ~15 min on A100 (20 samples × 15 conditions = 300 problems).

In [ ]:
!python eval/baseline_eval.py \
    --samples 20 \
    --output eval/baseline_results.json \
    --verbose \
    2>&1 | tee eval/baseline_log.txt

## Step 7 — Fetch OOD Evaluation Data

Downloads 50 medical (MMLU) + 50 legal (AGIEval LSAT) Q&A pairs.  
Run once — these become the transfer-evaluation test set.

In [ ]:
!python eval/ood/fetch_ood_data.py --n 50 2>&1

## Step 8 — Full GRPO Training

> ⏱️ Estimated time: ~60–90 min on A100, ~3–4 hrs on T4.

**What changed from old version:**
- ✅ `_GT_STORE` dict replaced with dataset columns → no tokeniser-drift reward misses
- ✅ `beta=0.1`, `max_grad_norm=0.5`, `lr=5e-6` → KL stable
- ✅ 4 orthogonal reward functions: Brier + Format + Accuracy + Anti-hedge
- ✅ KL early-stop callback (halts if KL > 0.5 for 20 consecutive steps)

Watch for `mean` reward climbing from ~−0.25 toward 0.0 as calibration improves.

In [ ]:
# Set one of: t4, l4, a100 (or none). Start with 200-500 steps, then scale.
COLAB_PROFILE = "t4"
MAX_STEPS = 300

# Remove --no-wandb if you set WANDB_API_KEY above
!python training/train_grpo.py \
  --no-wandb \
  --colab-profile {COLAB_PROFILE} \
  --max-steps {MAX_STEPS} \
  --save-steps 50 \
  --logging-steps 1 \
  2>&1 | tee training/grpo_training_log.txt

In [ ]:
# Optional: fully custom GRPO hyperparameters (uncomment to use)
# !python training/train_grpo.py \
#   --no-wandb \
#   --model-id Qwen/Qwen2.5-3B-Instruct \
#   --output-dir ./honest-qwen-3b-grpo \
#   --prompt-dataset-size 3000 \
#   --num-generations 8 \
#   --per-device-train-batch-size 1 \
#   --gradient-accumulation-steps 16 \
#   --max-completion-length 1024 \
#   --temperature 1.0 \
#   --learning-rate 1.5e-6 \
#   --beta 0.04 \
#   --max-grad-norm 1.0 \
#   --warmup-steps 10 \
#   --save-steps 50 \
#   --logging-steps 1 \
#   --seed 42 \
#   --max-steps 500 \
#   2>&1 | tee training/grpo_training_log_custom.txt

## Step 9 — Post-training Full Evaluation

In [ ]:
# Run full evaluation: in-dist + OOD + comparison table
!python eval/full_eval.py \
    --model-id Qwen/Qwen2.5-3B-Instruct \
    --adapter-path ./honest-qwen-3b-grpo/final_adapters \
    --baseline-results eval/baseline_results.json \
    --ood-dir eval/ood \
    --output eval/full_results.json \
    --samples 20 \
    2>&1 | tee eval/full_eval_log.txt

## Step 10 — Plot Before / After Reliability Diagram

In [ ]:
!pip install matplotlib seaborn --quiet
from eval.plot_reliability import plot_comparison

out = plot_comparison(
    'eval/baseline_results.json',
    'eval/full_results.json',
    label_before='Before GRPO (Qwen2.5-3B-Instruct)',
    label_after='After GRPO Training',
)
print(f'Comparison plot saved to: {out}')

# Display inline in Colab
from IPython.display import Image
Image(out)

## Step 11 — (Optional) Push Adapter to HuggingFace Hub

In [ ]:
HF_USERNAME  = 'your-username'   # ← update
HF_REPO_NAME = 'honest-qwen-3b-grpo'

!huggingface-cli login --token $HF_TOKEN
!cd honest-qwen-3b-grpo/final_adapters && \
    huggingface-cli upload {HF_USERNAME}/{HF_REPO_NAME} . .